# 08 | LangGraph 工具节点：tools、ToolNode、Runnable 是什么

前两篇已经看过固定边和条件边。接下来进入 Agent 里很常见的一步：工具调用。

工具调用的核心问题是：

```text
当模型决定要调用一个 Python 函数时，LangGraph 怎么把这个函数接进图里？
```

这篇只讲三个概念：`tools`、`ToolNode`、`Runnable`。

## 一、从一个工具调用流程开始

为了专心看工具节点，这里不接真实模型。我们手动构造一条 `AIMessage`，模拟“模型决定调用工具”的结果。

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode


# tool：把普通 Python 函数包装成可被模型/ToolNode 使用的工具。
@tool
def calculate_total(unit_price: float, quantity: int) -> str:
    """计算订单总价。"""
    return f"总价：{unit_price * quantity:.2f} 元"


# tools：工具列表。真实 Agent 中，模型绑定和 ToolNode 通常共用这份列表。
TOOLS = [calculate_total]


# Node：这里不调用真实模型，只手动生成一次工具调用请求。
def assistant(state: MessagesState) -> dict[str, list[AIMessage]]:
    tool_call = {
        "name": "calculate_total",
        "args": {"unit_price": 19.9, "quantity": 3},
        "id": "call_1",
    }
    return {"messages": [AIMessage(content="", tool_calls=[tool_call])]}


# StateGraph：MessagesState 用来保存 HumanMessage、AIMessage、ToolMessage。
builder = StateGraph(MessagesState)

builder.add_node("assistant", assistant)

# ToolNode：内置工具节点。它会读取 AIMessage.tool_calls，并执行对应工具。
builder.add_node("tools", ToolNode(TOOLS))

builder.add_edge(START, "assistant")
builder.add_edge("assistant", "tools")
builder.add_edge("tools", END)

graph = builder.compile()

result = graph.invoke(
    {"messages": [HumanMessage(content="3 件 19.9 元的商品，一共多少钱？")]}
)

for message in result["messages"]:
    print(type(message).__name__, "->", message.content)

这段代码里，执行链路是：

```text
HumanMessage
 -> assistant 生成 AIMessage.tool_calls
 -> ToolNode 执行 calculate_total
 -> 写回 ToolMessage
 -> END
```

真实 Agent 中，`assistant` 通常是模型节点，工具调用请求由模型生成。这里手写 `AIMessage.tool_calls`，只是为了让例子不依赖模型 API。

## 二、tool 和 tools：从函数到工具列表

`@tool` 来自 `langchain_core.tools`。它会把普通 Python 函数包装成工具对象。

```python
@tool
def calculate_total(unit_price: float, quantity: int) -> str:
    """计算订单总价。"""
    return f"总价：{unit_price * quantity:.2f} 元"
```

工具函数需要清楚写出参数类型，并且最好写 docstring。模型需要根据这些信息理解：这个工具能做什么、需要哪些参数。

工具本身也可以直接调用：

In [ ]:
calculate_total.invoke({"unit_price": 19.9, "quantity": 3})

这里的 `.invoke()` 也为后面理解 `Runnable` 做了铺垫。

### tools：工具列表

通常不会只定义一个工具，而是把工具放进列表：

```python
TOOLS = [calculate_total]
```

这个列表有两个常见用途：

| 用途 | 说明 |
| --- | --- |
| 绑定给模型 | 让模型知道有哪些工具可用 |
| 传给 `ToolNode` | 让 LangGraph 知道真正执行哪些 Python 函数 |

真实模型场景里，经常会看到类似写法：

```python
model = model.bind_tools(TOOLS)
builder.add_node("tools", ToolNode(TOOLS))
```

同一份 `TOOLS` 同时给模型和 `ToolNode` 使用，可以避免“模型知道一个工具，但图里没有执行器”的错位。

## 三、ToolNode：执行 AIMessage 里的工具调用

`ToolNode` 是 LangGraph 提供的预构建节点。当前项目环境中，它从 `langgraph.prebuilt` 导入：

```python
from langgraph.prebuilt import ToolNode
```

`ToolNode(TOOLS)` 做的事情可以理解成三步：

```text
读取最新 AIMessage 里的 tool_calls
 -> 根据 name 找到对应工具
 -> 执行工具，并把结果写成 ToolMessage
```

所以我们不需要自己写“解析 tool_calls、匹配函数、生成 ToolMessage”的节点逻辑。

### AIMessage.tool_calls 是什么

在这个例子里，`assistant` 手动返回了一条带工具调用的 `AIMessage`：

```python
tool_call = {
    "name": "calculate_total",
    "args": {"unit_price": 19.9, "quantity": 3},
    "id": "call_1",
}
return {"messages": [AIMessage(content="", tool_calls=[tool_call])]}
```

字段含义很直接：

| 字段 | 含义 |
| --- | --- |
| `name` | 要调用哪个工具 |
| `args` | 传给工具的参数 |
| `id` | 本次工具调用的标识 |

`ToolNode` 正是读取这部分内容来执行工具。

## 四、Runnable：统一的可调用对象接口

`Runnable` 可以先简单理解成 LangChain / LangGraph 里的“统一调用接口”。

很多对象都可以用 `.invoke()` 调用，比如工具和编译后的图：

In [ ]:
print(calculate_total.invoke({"unit_price": 19.9, "quantity": 3}))

result = graph.invoke(
    {"messages": [HumanMessage(content="3 件 19.9 元的商品，一共多少钱？")]}
)
print(result["messages"][-1].content)

这就是 `Runnable` 思想最实用的一点：不管底层是工具、模型、链，还是编译后的图，调用方式都尽量统一。

初学阶段先记住 `.invoke()` 就够了；异步调用、批处理、流式输出可以后面再展开。

## 五、图示与小结

自动图可以这样看：

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### 小结

| 名称 | 一句话理解 |
| --- | --- |
| `tool` | 把普通 Python 函数包装成可调用工具 |
| `tools` | 一组工具列表，常同时给模型和 `ToolNode` 使用 |
| `ToolNode` | LangGraph 内置节点，负责执行 `AIMessage.tool_calls` 里的工具调用 |
| `Runnable` | 统一可调用接口，常见入口是 `.invoke()` |

后面接真实模型时，模型负责决定要不要调用工具；`ToolNode` 负责把这个决定真正执行掉。

参考资料：

- [LangGraph Graph API overview](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [LangChain tools 文档](https://docs.langchain.com/oss/python/langchain/tools)